In [1]:
import random
from cgra import *
from kernels import *

In [2]:
# Global variables
CGRA_N_ROWS = 3
CGRA_N_COLS = 3
SIZE = 60

# Adress
first_addr = 20000

kernel_name = "benchmarks/compigra_kernel/blas/PCA/3x3/"
version = f"_out_{CGRA_N_COLS}_I{SIZE}_J{SIZE}_K{SIZE}"

In [3]:
# ------------------------------------
#           FUNCTIONS
# ------------------------------------
def configMemory(inputX, mu, Xc_golden, XcT_golden, C_golden, NI, NJ):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------            
    first_addr_inputX = first_addr
    first_addr_mu = first_addr_inputX + NI*NJ*4
    first_addr_Xc_golden = first_addr_mu + NJ*4
    first_addr_XcT_golden = first_addr_Xc_golden + NI*NJ*4
    first_addr_C_golden = first_addr_XcT_golden + NJ*NI*4

    config_vals = [[] for i in range(CGRA_N_COLS)]

    # void PCA(int inputX[NI][NJ], int mu[NJ], int Xc[NI][NJ], int XcT[NJ][NI], int C[NJ][NJ])

    if version == "_3_I24_J24_K24":
        config_vals[0] = [first_addr_inputX, first_addr_mu, first_addr_XcT_golden, first_addr_C_golden] # 0, 1, 3, 4
        config_vals[1] = [first_addr_XcT_golden] # 3
        config_vals[2] = [first_addr_Xc_golden, first_addr_Xc_golden] # 2, 2
    
    if version == "_3_I60_J60_K60":
        config_vals[0] = [first_addr_inputX, first_addr_mu, first_addr_XcT_golden, first_addr_C_golden] # 0, 1, 3, 4
        config_vals[1] = [first_addr_XcT_golden] # 3
        config_vals[2] = [first_addr_Xc_golden, first_addr_Xc_golden] # 2, 2

    addr_config_loads = [0 for i in range(CGRA_N_COLS)]
    for i in range(CGRA_N_COLS):
        kernel_add_memory_region(kernel_name, addr_config_loads[i], config_vals[i], version=version)
        if i < CGRA_N_COLS -1:
            addr_config_loads[i+1] = addr_config_loads[i] + len(config_vals[i])*4
            
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_inputX, inputX, version=version)
    kernel_add_memory_region(kernel_name, first_addr_mu, mu, version=version)
    kernel_add_memory_region(kernel_name, first_addr_Xc_golden, Xc_golden, version=version)
    kernel_add_memory_region(kernel_name, first_addr_XcT_golden, XcT_golden, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C_golden, C_golden, version=version)

    # Config data address for direct loads
    return addr_config_loads

def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

def runKernel(load_addrs, max_it=1000, pr=["ROUT","INST"], printVal=1):
    # Run kernel
    run(kernel_name, pr=pr, load_addrs=load_addrs, version=version, limit=max_it, printVal=printVal)

def getResult(first_addr, length):
    result = [0 for _ in range(length)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr) and (int(row[0]) < first_addr + length*4):
                    result[int((int(row[0]) - first_addr)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [4]:
# --------------------------------------------
#               DATA
# --------------------------------------------
data = np.load(kernel_name + f"data/data_{SIZE}x{SIZE}.npz")

inputX = data["inputX"]
mu = data["mu"]
Xc_golden = data["Xc_golden"]
XcT_golden = data["XcT_golden"]
C_golden = data["C_golden"]
expected_res = data["C_golden"].copy()
NI = int(data["NI"])
NJ = int(data["NJ"])


print(f"Testing PCA sizes : {NI}x{NJ}")

Testing PCA sizes : 24x24


In [5]:
load_addrs = configMemory(inputX, mu, Xc_golden, XcT_golden, C_golden, NI, NJ)

In [6]:
runKernel(load_addrs, max_it=2000000, printVal=0)
#estimatedConfigCycles(kernel_name, version)

Execution accurate cycles: 35351
Config cycles: 207
Total cycles: 35558
Memory cycles: 15011
Arithmetic cycles: 19739
END


In [7]:
# Get result from CGRA
first_addr_res = first_addr + NI*NJ*4*3 + NJ*4
result = getResult(first_addr_res, NI*NJ)

# Check result correctness
errors = 0
err_idx = []
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
        err_idx.append(i)
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(expected_res, NI, NJ)
    print("CGRA: ")
    printAsMatrix(result, NI, NJ)
    print("Errors are: Exp : CGRA")
    for i in err_idx:
        print(f"Idx[{i}] {expected_res[i]} : {result[i]}")

else:
    print("OK")



OK
